## Imports

In [ ]:
import datasets
import os
import pandas as pd
from itertools import islice
from dotenv import load_dotenv
from tqdm import tqdm
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
load_dotenv()
datasets.logging.set_verbosity_error()
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from common.helper import clickhouse_extract_data_as_df
from common.const import TABLE_RAW_PRODUCTS,TABLE_RAW_REVIEWS,SCHEMA

cpu


## Get Data from ClickHouse

### Products Data

In [6]:
categories = [
    "Cell Phones & Accessories",
    "Toys & Games",
    "All Electronics",
    "Computers"
]
dfs = []

for category in categories:
    query = f"""
    SELECT
        parent_asin,
        main_category,
        title,
        features,
        description
    FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
    where main_category = '{category}'
    order by rand()
    LIMIT 5000
    """

    df = clickhouse_extract_data_as_df(query)
    dfs.append(df)

products_df = pd.concat(dfs)

In [8]:
products_df.shape
## should be (20,000,4)

(20000, 5)

### Create Item Metadata

In [24]:
def concat_item_metadata(dp):
    meta = ''
    flag = False

    if dp['title'] is not None:
        meta += dp['title']
        flag = True

    if dp['features'] is not None and dp['features'] is not pd.NA:
        if len(dp['features']) > 0:
            if flag:
                meta += ' '
            meta += ''.join(dp['features'])
            flag = True

    if dp['description'] is not None and dp['description'] is not pd.NA:
        if len(dp['description']) > 0:
            if flag:
                meta += ' '
            meta += ''.join(dp['description'])

    dp['cleaned_metadata'] = meta \
        .replace('\t', ' ') \
        .replace('\n', ' ') \
        .replace('\r', '') \
        .strip()

    return dp

In [ ]:
products_df_cleaned = products_df.apply(concat_item_metadata, axis=1)
products_df_cleaned['cleaned_metadata'].iloc[3]

'MyBat IPTCH4HPCTUFFIM009NP TUFF Hybrid Phone Protector Cover for iPod Touch 4 (Dots Pink/White) A soft rubberized coating tightly grips to your phone while absorbing shocks and bumps. Exterior layer is a hardened shell with a sleek smooth finish. Reinforces vulnerable corners and adds additional side grips for better control of your phone. Mybat iPod touch (4th generation) Dots(Pink/white)/White TUFF Hybrid Phone Protector Cover'

### User Data

In [ ]:
product_ids = products_df["parent_asin"].unique().tolist()
len(product_ids)

4000

In [ ]:
asin_list = ",".join([f"'{a}'" for a in product_ids])

reviews_query = f"""
SELECT
    user_id,
    parent_asin,
    title,
    text,
    rating,
    timestamp
FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
WHERE parent_asin IN ({asin_list})
LIMIT 4000
"""
user_review_df = clickhouse_extract_data_as_df(reviews_query)

In [ ]:
user_review_df.shape

(4000, 6)

### Create User Metadata

In [ ]:
def concat_review(dp):
    review = ''
    flag = False
    if dp['title'] is not None:
        review += dp['title']
        flag = True
    if dp['text'] is not None:
        if flag:
            review += ' '
        review += dp['text']
    dp['cleaned_review'] = review \
        .replace('\t', ' ') \
        .replace('\n', ' ') \
        .replace('\r', '') \
        .strip()
    return dp

In [ ]:
user_review_df_cleaned = user_review_df.apply(concat_review,axis =1)

In [ ]:
user_review_df_cleaned.shape

(4000, 7)

## Vectorize and Store in Pinecone

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("hyp1231/blair-roberta-large")
model = AutoModel.from_pretrained("hyp1231/blair-roberta-large")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

#### Connect to Pinecone

In [ ]:
from pinecone import Pinecone
pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])
index = pc.Index("scc454")

#### Vectorize Function

In [31]:
products_df_cleaned = products_df_cleaned.reset_index(drop=True)

def vectorize_batched(dataframe, column_name, batch_size=32):
    model.eval()

    model.to(device)

    embeddings_list = []

    texts = dataframe[column_name].fillna("").astype(str).tolist()

    for i in tqdm(range(0, len(texts), batch_size), desc="Vectorizing batches"):
        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
        inputs = {k: v.to(device) for k,v in inputs.items()}  # move to TPU

        with torch.no_grad():
            batch_embeddings = model(**inputs, return_dict=True).last_hidden_state[:, 0]
            batch_embeddings = batch_embeddings / batch_embeddings.norm(dim=1, keepdim=True)

        # Move back to CPU for concatenation
        embeddings_list.append(batch_embeddings.cpu())

        if (i // batch_size) % 10 == 0:
            print(f"Processed batch {i // batch_size + 1} / {len(texts) // batch_size + 1}")

    embeddings = torch.cat(embeddings_list, dim=0)
    return embeddings

#### Batch Upload Function

In [35]:
def upsertinBatches(index,products_df,semantic_embeddings,num_batches=200):
    import numpy as np
    data_for_upsert = []
    for i, row in products_df.iterrows():
        data_for_upsert.append({
            "id": str(row['parent_asin']),
            "values": semantic_embeddings[i].tolist(),
            "metadata": {
                "title": str(row['title'])[:500], # Metadata limit: truncate if too long
                "main_category": str(row['main_category']),

            }
        })

    for i in range(0, len(data_for_upsert), num_batches):

        batch = data_for_upsert[i : i + num_batches]

        index.upsert(vectors=batch, namespace='item')

        if (i + num_batches) % 1000 == 0 or (i + num_batches) >= len(data_for_upsert):
            print(f"Upserted {min(i + num_batches, len(data_for_upsert))} / {len(data_for_upsert)} vectors")

    print("Success: All batches uploaded to Pinecone.")


#### Vectorize and Upload

In [33]:
embeddings = vectorize_batched(products_df_cleaned,"cleaned_metadata")
print(embeddings.shape)

Vectorizing batches:   0%|          | 1/625 [00:03<32:01,  3.08s/it]

Processed batch 1 / 626


Vectorizing batches:   2%|▏         | 11/625 [00:36<34:46,  3.40s/it]

Processed batch 11 / 626


Vectorizing batches:   3%|▎         | 21/625 [01:08<31:56,  3.17s/it]

...

Processed batch 611 / 626


Vectorizing batches:  99%|█████████▉| 621/625 [33:15<00:12,  3.24s/it]

Processed batch 621 / 626


Vectorizing batches: 100%|██████████| 625/625 [33:28<00:00,  3.21s/it]

torch.Size([20000, 1024])


In [36]:
upsertinBatches(index,products_df_cleaned,embeddings)

Upserted 1000 / 20000 vectors
Upserted 2000 / 20000 vectors
Upserted 3000 / 20000 vectors
Upserted 4000 / 20000 vectors
Upserted 5000 / 20000 vectors
Upserted 6000 / 20000 vectors
Upserted 7000 / 20000 vectors
Upserted 8000 / 20000 vectors
Upserted 9000 / 20000 vectors
Upserted 10000 / 20000 vectors
Upserted 11000 / 20000 vectors
Upserted 12000 / 20000 vectors
Upserted 13000 / 20000 vectors
Upserted 14000 / 20000 vectors
Upserted 15000 / 20000 vectors
Upserted 16000 / 20000 vectors
Upserted 17000 / 20000 vectors
Upserted 18000 / 20000 vectors
Upserted 19000 / 20000 vectors
Upserted 20000 / 20000 vectors
Success: All batches uploaded to Pinecone.


#### Upsert User Embeddings

In [ ]:
user_embeddings = vectorize_batched(user_review_df_cleaned,"cleaned_review")

Vectorizing batches:   1%|          | 1/125 [00:01<02:43,  1.32s/it]

Processed batch 1 / 126


Vectorizing batches:   9%|▉         | 11/125 [00:12<01:45,  1.08it/s]

Processed batch 11 / 126


Vectorizing batches:  17%|█▋        | 21/125 [00:23<02:00,  1.16s/it]

...

Vectorizing batches:  89%|████████▉ | 111/125 [02:52<00:29,  2.12s/it]

Processed batch 111 / 126


Vectorizing batches:  97%|█████████▋| 121/125 [03:00<00:03,  1.33it/s]

Processed batch 121 / 126


Vectorizing batches: 100%|██████████| 125/125 [03:03<00:00,  1.47s/it]


In [ ]:
def upsertUserinBatches(index,products_df,semantic_embeddings,num_batches=200):
    import numpy as np
    data_for_upsert = []
    for i, row in products_df.iterrows():
        data_for_upsert.append({
            "id": str(row['user_id']),
            "values": semantic_embeddings[i].tolist(),
            "metadata": {
                "title": str(row['title'])[:500], # Metadata limit: truncate if too long
            }
        })

    for i in range(0, len(data_for_upsert), num_batches):

        batch = data_for_upsert[i : i + num_batches]

        index.upsert(vectors=batch, namespace='user_reviews')

        if (i + num_batches) % 1000 == 0 or (i + num_batches) >= len(data_for_upsert):
            print(f"Upserted {min(i + num_batches, len(data_for_upsert))} / {len(data_for_upsert)} vectors")

    print("Success: All batches uploaded to Pinecone.")


In [ ]:
upsertUserinBatches(index,user_review_df_cleaned,user_embeddings)

Upserted 1000 / 4000 vectors
Upserted 2000 / 4000 vectors
Upserted 3000 / 4000 vectors
Upserted 4000 / 4000 vectors
Success: All batches uploaded to Pinecone.


## Evaluation

In [ ]:
all_product_ids = []

for ids in index.list(namespace='item'):
    all_product_ids.extend(ids)

print(len(all_product_ids))

20000


In [9]:
all_user_ids = []

for ids in index.list(namespace='user_reviews'):
    all_user_ids.extend(ids)

print(len(all_user_ids))

3923


In [ ]:
def recall_at_k(ground_truth, predictions, k=10):
    if len(ground_truth) == 0:
        return 0.0

    top_k_preds = predictions[:k]

    hits = len(set(top_k_preds) & set(ground_truth))

    recall = hits / len(ground_truth)
    return recall

In [10]:
user_list = ",".join([f"'{a}'" for a in all_user_ids])
asin_list = ",".join([f"'{a}'" for a in all_product_ids])

valid_query = f"""
SELECT
    *
FROM {SCHEMA}.5core_timestamp_w_his_valid
WHERE user_id IN ({user_list})
  AND parent_asin IN ({asin_list})
  AND rating == 5
"""
user_valid_df = clickhouse_extract_data_as_df(valid_query)

In [11]:
ground_truths = user_valid_df['parent_asin']
ground_truths.shape

(2,)

### Getting User Reviews to Generate Search Query using LLM's

In [12]:
ground_truths_str = ",".join([f"'{a}'" for a in ground_truths])
review_query = f"""
SELECT
    parent_asin,
    title,
    text,
    rating
FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
WHERE parent_asin IN ({ground_truths_str})
  AND rating = 5
  AND lengthUTF8(text) > 100
"""
eval_review_query = clickhouse_extract_data_as_df(review_query)
eval_review_query.shape

(2069, 4)

In [14]:
eval_review_query.iloc[2]

parent_asin                                           B08K2S5B88
title          Great for the gym - Husband Loves them. Kind o...
text           These are great! Good sound, easy to use and l...
rating                                                         5
Name: 2, dtype: object

In [15]:
ground_truths

0    B0BDLHN5LZ
1    B08K2S5B88
Name: parent_asin, dtype: string

### Using GEMINI to convert the review into a user search

In [16]:
SYSTEM_PROMPT = """You are an assistant that converts user product reviews into complex, conversational product search queries.
Your task is to read a 5-star user review and rephrase it into a detailed natural language context using a first-person tone. Imagine you are the user speaking to a highly advanced search engine, explaining exactly what you want, why you need it, and how it will help you.
Rules:
Use a first-person tone: Always write from the perspective of the user (e.g., "I want a...", "I need something that...").
Keep the context and motivation: Do not just list physical features. Include the user's background, their motivations, the problem they are trying to solve, and their emotional state or feelings about the desired outcome.
Be conversational and complex: The output should read like a natural, human explanation rather than a robotic list of keywords.
Do not include any explanations, reasoning, or extra text outside of the generated query.
Example 1: Review: "This salt gun is incredible for my garden. I get so many stink bugs and spiders in my house and I absolutely hate them. This thing works perfectly and I don't have to get close to them anymore." Search Query: I want a gun that I can use while gardening to get rid of stink bugs, ants, flies, and spiders in my house. It needs to be amazing and help me feel less scared.
Example 2: Review: "This puzzle is a lot of fun and quite challenging. Each mini-fig head is made from multiple pieces which makes it difficult but enjoyable. I would definitely recommend it for Lego fans." Search Query: I am a massive Lego fan looking for something fun but difficult to put together. I want a challenging puzzle that features a bunch of different mini-figure expressions where the heads are made of multiple pieces to really test my skills."""

In [18]:
from google import genai
from google.genai import types

client = genai.Client()

def convert_to_search_query(x):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        config=types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT),
        contents = eval_review_query['title'].iloc[x] + " " + eval_review_query['text'].iloc[x]
    )

    return response.text

In [ ]:
user_search = convert_to_search_query(2)
print(user_search)
inputs = tokenizer(user_search, padding=True, truncation=True, max_length=512, return_tensors="pt")
inputs = {k: v.to(device) for k,v in inputs.items()}  # move to TPU

with torch.no_grad():
    embeddings = model(**inputs, return_dict=True).last_hidden_state[:, 0]
    embeddings = embeddings / embeddings.norm(dim=1, keepdim=True)

print(embeddings[0].shape)

I'm on the hunt for some fantastic headphones that I can confidently take to the gym. I absolutely need them to deliver really good sound, be intuitive and super easy to use, and feel genuinely lightweight on my head so they don't get in the way of my workout. I'm really keen on finding something that offers excellent value for the money, but without compromising on quality at all. It would be amazing if they came with their own protective case and could fold down nicely for compact storage. My husband loves the ones I got for him, but I do find my own pair a little on the bulky side compared to brands like Beats, so I'm hoping to discover something a bit more streamlined, yet still practical and robust for active use.
torch.Size([1024])


### Search for top-K in pinecone

In [36]:
top_k = 5 # number of similar items to retrieve
query_vector = embeddings[0].cpu().numpy().tolist()
results = index.query(
    vector=query_vector,
    top_k=top_k,
    include_values=True,   # optional: include vector in results
    include_metadata=True,  # optional: include metadata like title, ASIN
    namespace='item'
)

ids = [match.id for match in results.matches]
print(ids)

['B0BZ57TZWL', 'B082ZP889B', 'B01MYWM74V', 'B07CVGVK4S', 'B015EGRKK2']


In [22]:
query = f"""
    SELECT
        parent_asin,
        main_category,
        title,
        features,
        description
    FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
    where parent_asin in ({ids})
    order by parent_asin
    """
eval_data = clickhouse_extract_data_as_df(query)

### Using LLM-as-a Judge

In [23]:
system_prompt2 = """Your task is to evaluate how well a returned product matches a user query or intended use. You are given:

1. A **user query** describing what the user wants or needs from a product.
2. A **product description** including title, features, and specifications.

Your goal is to determine how effectively this product satisfies the user’s intent.

Rules:
1. Focus only on whether the product meets the functional requirements and desired features expressed in the user query.
2. Consider attributes, specifications, size, capacity, usability, and other relevant features.
3. Ignore filler language, marketing phrases, and irrelevant details.
4. Provide a clear **evaluation score** from 0 to 5, where:
   - 0 = Not relevant at all / does not satisfy the user
   - 1 = Barely satisfies any part of the user query
   - 2 = Partially satisfies some aspects
   - 3 = Moderately satisfies, but missing some key features
   - 4 = Mostly satisfies the user query
   - 5 = Fully satisfies the user query / excellent match
5. Optionally, provide 1-2 short sentences explaining your reasoning, but do not exceed 30 words.
6. Output only JSON in the following format:

{
  "score": <0-5>,
  "reason": "<short explanation>"
}

Example:

User Query:
"I need a rice strainer that can measure grains, rinse them easily, and drain water efficiently in a compact, dishwasher-safe design."

Product Description:
"Talisman Designs 2-in-1 Measure Rinse & Strain | Holds up to 2 Cups | Food Strainer | Dishwasher Safe | Compact Design for Kitchen"

Output:
{
  "score": 5,
  "reason": "Matches all key requirements: measures, rinses, drains, compact, dishwasher-safe."
}"""

In [25]:
eval_cleaned_data = eval_data.apply(concat_item_metadata,axis=1)

In [45]:

def evaluate_product(user_query, product_description, system_prompt):
    contents = f"User Query: {user_query}\nProduct Description: {product_description}"

    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        config=types.GenerateContentConfig(
            system_instruction=system_prompt
        ),
        contents=contents
    )

    # Convert text response to dictionary (assuming AI outputs JSON)
    import json
    try:
        evaluation = json.loads(response.text)
    except json.JSONDecodeError:
        evaluation = {"score": None, "reason": response.text}

    return evaluation

In [46]:
for x in range(len(eval_cleaned_data)):
    print(evaluate_product(user_query=user_search,product_description=eval_cleaned_data['cleaned_metadata'].iloc[x],system_prompt=system_prompt2))

{'score': 2, 'reason': 'These over-ear headphones are likely too bulky for the gym, lack a confirmed protective case, and appear to be wired, which is impractical for active workouts.'}
{'score': 4, 'reason': 'Matches sound quality, ease of use, and compact foldable design. However, it lacks a protective case and specific sweat-resistance ratings for the gym.'}
{'score': 3, 'reason': 'Matches folding, wireless, and ease-of-use needs, but fails to mention a protective case and lacks explicit confirmation of being lightweight/streamlined for the gym.'}
{'score': 3, 'reason': 'Matches foldable and wireless requirements, but lacks a protective case and the streamlined, high-quality feel the user specified for a non-bulky gym experience.'}
{'score': 4, 'reason': 'Matches sound quality, easy controls, and foldability. However, it lacks a mentioned protective case and explicit sweat-resistance for gym use.'}
{'score': 3, 'reason': "Meets sound, folding, and budget requirements, but lacks the 

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3-flash\nPlease retry in 16.597954582s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3-flash'}, '